<a href="https://colab.research.google.com/github/laisalegri/curso-fiap-agentes-autonomos-ia/blob/main/FIAP_hands_on_2_agente_atendimento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain -q
!pip install langchain-core -q
!pip install langchain-community -q
!pip install langchain-ollama -q
!pip install langgraph -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
!pip install -U langchain-ollama

In [3]:
!pgrep -af ollama
!curl -s http://127.0.0.1:11434/api/tags

In [4]:
# 1. Instala o zstd e depois instala o Ollama
!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Inicia o servidor do Ollama em background
!nohup ollama serve > /dev/null 2>&1 &

# 3. Faz o download do modelo
!ollama pull gpt-oss:20b

In [5]:
from langchain_community.llms import Ollama

def processamento_resposta(pergunta):
  # Initialize the Ollama LLM
  llm = Ollama(model="gpt-oss:20b",num_predict=1024)

  # Send a prompt
  resposta = llm.invoke(pergunta)
  return resposta

processamento_resposta("Diga: Hello World!")


/tmp/ipykernel_4002/3451206047.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.llms import Ollama
/tmp/ipykernel_4002/3451206047.py:5: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="gpt-oss:20b",num_predict=1024)


'Hello World!'

In [6]:
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, AIMessage
from typing_extensions import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langchain_ollama import ChatOllama

In [7]:
# Definir o schema do estado
class State(TypedDict):
    messages: Annotated[list, add_messages]

# Inicializar o modelo
llm = ChatOllama(model="gpt-oss:20b", temperature=0)

# Nó de entrada
def entrada_usuario(state: State) -> State:
     pergunta = input("Pergunta: ")
     if isinstance(pergunta, str) and pergunta.strip():
         return {"messages": [HumanMessage(content=pergunta)]}
     else:
         raise ValueError("A pergunta deve ser uma string não vazia.")

# Nó de processamento usando ChatOllama
def processar_solicitacao(state: State) -> State:
     last_message = state["messages"][-1]

     if isinstance(last_message, HumanMessage) and last_message.content.strip():
         print(f"DEBUG: Processando pergunta: {last_message.content}")
         try:
             resposta = llm.invoke([last_message]).content
             print(f"DEBUG: Resposta gerada: {resposta}")
             return {"messages": [AIMessage(content=resposta)]}
         except Exception as e:
             print(f"DEBUG: Erro ao chamar o LLM: {e}")
             return {"messages": [AIMessage(content=f"Desculpe, ocorreu um erro: {e}")]}

     else:
         raise ValueError("A última mensagem no estado não é uma HumanMessage válida.")

# Nó de saída
def saida_resposta(state: State) -> State:
     last_message = state["messages"][-1]
     if hasattr(last_message, 'content'):
         print(f"Resposta: {last_message.content}")
     else:
         print(f"Resposta: {last_message}")
     return state

# Criar o grafo de estados
grafo = StateGraph(State)
grafo.add_node("entrada", entrada_usuario)
grafo.add_node("processamento", processar_solicitacao)
grafo.add_node("saida", saida_resposta)

grafo.add_edge(START, "entrada")
grafo.add_edge("entrada", "processamento")
grafo.add_edge("processamento", "saida")
grafo.add_edge("saida", END)

compiled = grafo.compile()

# Loop de interação
print("Iniciando a interação...")
while True:
     try:
         compiled.invoke({"messages": []})
         continuar = input("Deseja fazer outra pergunta? (sim/não): ")
         if continuar.lower() != "sim":
             break
     except ValueError as ve:
         print(f"Erro de entrada: {ve}")
     except KeyboardInterrupt:
         print("\nInteração encerrada pelo usuário.")
         break
     except Exception as e:
         print(f"Ocorreu um erro inesperado: {e}")
         break

print("Interação encerrada.")

Iniciando a interação...
Pergunta: Qual é a capital de Minas Gerais?
DEBUG: Processando pergunta: Qual é a capital de Minas Gerais?
DEBUG: Resposta gerada: A capital de Minas Gerais é **Belo Horizonte**.
Resposta: A capital de Minas Gerais é **Belo Horizonte**.
Deseja fazer outra pergunta? (sim/não): não
Interação encerrada.


# Análise de Sentimento

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage

# Inicializa o modelo ChatOllama
llm = ChatOllama(model="gpt-oss:20b", temperature=0)


def analisar_sentimento(comentario: str) -> str:
    """
    Analisa o sentimento de um comentário usando o LLM.
    Retorna 'POSITIVO', 'NEGATIVO' ou 'NEUTRO'.
    """
    prompt = (
        f"Classifique o sentimento do seguinte comentário como POSITIVO, NEGATIVO ou NEUTRO:\n"
        f"Comentário: \"{comentario}\""
    )

    # Chamando o LLM com a mensagem do usuário
    resposta = llm.invoke([HumanMessage(content=prompt)])

    # Retorna apenas o texto da resposta
    return resposta.content.strip().upper()


# Teste do sistema com um comentário de rede social
while True:
    comentario = input()
    sentimento = analisar_sentimento(comentario)
    print(f"O sentimento do comentário é: {sentimento}")

essa comida é meio estranha, mas eu gostei do atendimento
O sentimento do comentário é: NEUTRO
a comida é muito boa, voltarei mais vezes
O sentimento do comentário é: POSITIVO
não recomendo, demorou muito para servir
O sentimento do comentário é: NEGATIVO
meu arroz estava com gosto de barata
O sentimento do comentário é: NEGATIVO
minha carne estava dura
O sentimento do comentário é: NEGATIVO
puta que pariu que comida gostosa do caralho vai tomar no cu
O sentimento do comentário é: NEGATIVO
puta que pariu que comida gostosa do caralho
O sentimento do comentário é: POSITIVO
eu não sei se a comida estava boa
O sentimento do comentário é: NEUTRO
a comida transcendeu a realidade é como se vários anjos me abraçassem, é como se o cosmos me agradecesse por existir, depois dessa experiência nunca mais serei o mesmo
O sentimento do comentário é: POSITIVO
gostei da comida, não é das melhores que já comi, mas valeu o preço
O sentimento do comentário é: POSITIVO
não é das melhores que já comi, mas 